# Optimized Deep Learning Framework for Multi-Class CT-Based Stroke Diagnosis
## 3-Class Classification: Normal vs. Hemorrhagic vs. Ischemic
---
**Enhanced Research Edition**: Featuring Focal Loss, EfficientNetV2, ConvNeXt, and Grad-CAM Interpretability.

**Datasets**:
1. **Intracranial Hemorrhage (ICH)**: Computed Tomography Images for Intracranial Hemorrhage Detection.
2. **Acute Ischemic Stroke (CPAISD)**: CPAISD dataset converted from DICOM.

**Author**: Jules (AI Research Engineer assistant)

In [ ]:
!pip install -q pydicom opencv-python-headless scipy

In [ ]:
import os, random, warnings, glob, shutil, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
import pydicom
from pathlib import Path
from datetime import datetime
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    f1_score, precision_score, recall_score, accuracy_score, roc_auc_score
)

import tensorflow as tf
from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow {tf.__version__} | GPUs: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
class Config:
    BASE_DIR = Path.cwd()
    
    # Data paths (Kaggle structure)
    KAGGLE_ICH_ROOT = Path("/kaggle/input/datasets/vbookshelf/computed-tomography-ct-images/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.0.0")
    KAGGLE_CPAISD   = Path("/kaggle/input/datasets/orvile/cpaisd-acute-ischemic-stroke-dataset/dataset")
    
    # Fallback to local if not on Kaggle
    ICH_ROOT = KAGGLE_ICH_ROOT if KAGGLE_ICH_ROOT.exists() else BASE_DIR / "ich_dataset"
    CPAISD_ROOT = KAGGLE_CPAISD if KAGGLE_CPAISD.exists() else BASE_DIR / "cpaisd_dataset"
    
    ICH_CSV = next(ICH_ROOT.rglob("hemorrhage_diagnosis.csv"), None)
    ICH_IMGS = next(ICH_ROOT.rglob("ct_scans"), ICH_ROOT)
    
    DICOM_DIR = BASE_DIR / "dicom_converted"
    OUTPUT_DIR = BASE_DIR / "outputs"
    MODEL_DIR = OUTPUT_DIR / "models"
    PLOT_DIR = OUTPUT_DIR / "plots"
    
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 32
    EPOCHS_STAGE1 = 15
    EPOCHS_STAGE2 = 10
    LR_STAGE1 = 1e-4
    LR_STAGE2 = 1e-5
    
    CLASS_NAMES = {0: "Normal", 1: "Hemorrhagic", 2: "Ischemic"}
    N_CLASSES = 3

for d in [Config.DICOM_DIR, Config.MODEL_DIR, Config.PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration initialized.")

## Data Utilities

In [ ]:
def dicom_to_png(dcm_path, out_path, wl=40, ww=80):
    """Converts DICOM to PNG with Brain Windowing."""
    try:
        dcm = pydicom.dcmread(str(dcm_path))
        arr = dcm.pixel_array.astype(np.float32)
        slope = float(getattr(dcm, "RescaleSlope", 1))
        inter = float(getattr(dcm, "RescaleIntercept", 0))
        arr = arr * slope + inter
        lo, hi = wl - ww/2, wl + ww/2
        arr = np.clip(arr, lo, hi)
        arr = ((arr - lo) / (hi - lo) * 255).astype(np.uint8)
        rgb = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_path), rgb)
        return True
    except Exception:
        return False

def convert_ischemic_dataset(cp_root, out_dir):
    cp_root = Path(cp_root)
    out_dir = Path(out_dir)
    dcm_files = list(cp_root.rglob("*.dcm"))
    print(f"Found {len(dcm_files)} DICOM files in CPAISD.")
    count = 0
    for dcm_p in dcm_files:
        if any(x in str(dcm_p).lower() for x in ["seg", "mask", "label"]): continue
        rel = dcm_p.relative_to(cp_root)
        out_name = "__".join(rel.parts).replace(".dcm", ".png")
        if dicom_to_png(dcm_p, out_dir / out_name):
            count += 1
    print(f"Converted {count} Ischemic PNGs.")

def load_ich_dataset():
    def _norm_col(c): return re.sub(r"[^a-z0-9]", "", c.lower())
    
    label_map = {}
    if Config.ICH_CSV and Config.ICH_CSV.exists():
        df_csv = pd.read_csv(Config.ICH_CSV)
        col_map = {_norm_col(c): c for c in df_csv.columns}
        p_col, s_col, n_col = col_map.get("patientnumber"), col_map.get("slicenumber"), col_map.get("nohemorrhage")
        if p_col and s_col and n_col:
            for _, r in df_csv.iterrows():
                label_map[(int(r[p_col]), int(r[s_col]))] = 0 if int(r[n_col]) == 1 else 1
    
    imgs = [p for p in Config.ICH_IMGS.rglob("*.jpg") if "mask" not in p.name.lower()]
    records = []
    for p in imgs:
        nums = [int(x) for x in re.findall(r"\d+", str(p))]
        label = None
        # Attempt to match by patient/slice numbers in path
        for i in range(len(nums)-1):
            if (nums[i], nums[i+1]) in label_map:
                label = label_map[(nums[i], nums[i+1])]
                break
        if label is not None:
            records.append({"filepath": str(p), "patient_id": f"ICH_{nums[0]}", "label": label, "source": "ICH"})
    
    return pd.DataFrame(records)

def load_ischemic_dataset():
    pngs = list(Config.DICOM_DIR.glob("*.png"))
    records = []
    for p in pngs:
        parts = p.stem.split("__")
        pid = parts[1] if len(parts) > 1 else "Unknown"
        records.append({"filepath": str(p), "patient_id": f"ISC_{pid}", "label": 2, "source": "CPAISD"})
    return pd.DataFrame(records)

def get_dataset():
    if not any(Config.DICOM_DIR.glob("*.png")) and Config.CPAISD_ROOT.exists():
        convert_ischemic_dataset(Config.CPAISD_ROOT, Config.DICOM_DIR)
    
    df_ich = load_ich_dataset()
    df_isc = load_ischemic_dataset()
    df = pd.concat([df_ich, df_isc], ignore_index=True).drop_duplicates(subset="filepath")
    print(f"Total Images: {len(df)}")
    print(df.label.value_counts())
    return df

In [ ]:
def patient_wise_split(df, test_frac=0.15, val_frac=0.15):
    pl = df.groupby("patient_id")["label"].agg(lambda x: x.mode()[0]).reset_index()
    p_train, p_temp = train_test_split(pl.patient_id, test_size=test_frac+val_frac, stratify=pl.label, random_state=SEED)
    p_val, p_test = train_test_split(p_temp, test_size=test_frac/(test_frac+val_frac), random_state=SEED)
    
    return (
        df[df.patient_id.isin(p_train)].reset_index(drop=True),
        df[df.patient_id.isin(p_val)].reset_index(drop=True),
        df[df.patient_id.isin(p_test)].reset_index(drop=True)
    )

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.05),
], name="augmentation")

def parse_fn(fp, label, training=False):
    img = tf.io.read_file(fp)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, Config.IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    if training:
        img = augmentation(img, training=True)
    return img, tf.cast(label, tf.int32)

def make_ds(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df.filepath.values, df.label.values))
    if training: ds = ds.shuffle(len(df), seed=SEED)
    ds = ds.map(lambda x, y: parse_fn(x, y, training), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(Config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## Advanced Models & Loss Functions

In [ ]:
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=0.25, gamma=2.0, name="focal_loss"):
        super().__init__(name=name)
        self.alpha = alpha
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=Config.N_CLASSES)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        loss = -y_true * (self.alpha * tf.pow((1 - y_pred), self.gamma) * tf.math.log(y_pred))
        return tf.reduce_sum(loss, axis=-1)

In [ ]:
def build_densenet121():
    base = tf.keras.applications.DenseNet121(include_top=False, weights="imagenet", input_shape=Config.IMG_SIZE + (3,))
    base.trainable = False
    inp = Input(shape=Config.IMG_SIZE + (3,))
    x = tf.keras.applications.densenet.preprocess_input(inp * 255.0)
    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(Config.N_CLASSES, activation="softmax")(x)
    return Model(inp, out, name="DenseNet121"), base

def build_resnet50v2():
    base = tf.keras.applications.ResNet50V2(include_top=False, weights="imagenet", input_shape=Config.IMG_SIZE + (3,))
    base.trainable = False
    inp = Input(shape=Config.IMG_SIZE + (3,))
    x = tf.keras.applications.resnet_v2.preprocess_input(inp * 255.0)
    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(Config.N_CLASSES, activation="softmax")(x)
    return Model(inp, out, name="ResNet50V2"), base

def build_efficientnetv2s():
    base = tf.keras.applications.EfficientNetV2S(include_top=False, weights="imagenet", input_shape=Config.IMG_SIZE + (3,))
    base.trainable = False
    inp = Input(shape=Config.IMG_SIZE + (3,))
    # EfficientNetV2 internal rescaling handles input normalization automatically if needed
    x = base(inp)
    x = layers.GlobalAveragePooling2D()(x)
    out = layers.Dense(Config.N_CLASSES, activation='softmax')(x)
    return Model(inp, out, name="EfficientNetV2S"), base

def build_convnext_tiny():
    base = tf.keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet", input_shape=Config.IMG_SIZE + (3,))
    base.trainable = False
    inp = Input(shape=Config.IMG_SIZE + (3,))
    x = base(inp)
    x = layers.GlobalAveragePooling2D()(x)
    out = layers.Dense(Config.N_CLASSES, activation="softmax")(x)
    return Model(inp, out, name="ConvNeXtTiny"), base

BUILDERS = {
    "DenseNet121": build_densenet121,
    "ResNet50V2": build_resnet50v2,
    "EfficientNetV2S": build_efficientnetv2s,
    "ConvNeXtTiny": build_convnext_tiny
}

## Interpretability & Evaluation Utilities

In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    grad_model = Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None: pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
        
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.resize(img, Config.IMG_SIZE)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = heatmap * alpha + img
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title("Original")
    plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)); plt.title("Grad-CAM")
    plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=Config.CLASS_NAMES.values(), 
                yticklabels=Config.CLASS_NAMES.values())
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.savefig(Config.PLOT_DIR / f"cm_{model_name}.png")
    plt.show()

def plot_roc_auc(y_true, y_probs, model_name):
    plt.figure(figsize=(8, 6))
    for i in range(Config.N_CLASSES):
        fpr, tpr, _ = roc_curve(y_true == i, y_probs[:, i])
        plt.plot(fpr, tpr, label=f"{Config.CLASS_NAMES[i]} (AUC = {auc(fpr, tpr):.2f})")
    
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f"ROC-AUC - {model_name}")
    plt.legend()
    plt.savefig(Config.PLOT_DIR / f"roc_{model_name}.png")
    plt.show()

## Training Pipeline & Ensemble

In [ ]:
def train_model(model_name, train_ds, val_ds):
    print(f"\n{'='*20}\nTraining {model_name}\n{'='*20}")
    model, base = BUILDERS[model_name]()
    
    # Stage 1: Warm-up (Head only)
    model.compile(optimizer=Adam(Config.LR_STAGE1), loss=FocalLoss(), metrics=["accuracy"])
    cbs = [
        EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
    ]
    
    print("Stage 1: Head Warm-up")
    model.fit(train_ds, validation_data=val_ds, epochs=Config.EPOCHS_STAGE1, callbacks=cbs)
    
    # Stage 2: Fine-tuning (Unfreeze some layers)
    print("Stage 2: Fine-tuning")
    base.trainable = True
    # For simplicity, unfreeze the whole base, or you can specify layers
    model.compile(optimizer=Adam(Config.LR_STAGE2), loss=FocalLoss(), metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=Config.EPOCHS_STAGE2, callbacks=cbs)
    
    model.save(Config.MODEL_DIR / f"{model_name}.keras")
    return model

def evaluate_ensemble(models, test_ds, y_true):
    all_probs = []
    model_f1s = []
    
    # Get weights based on F1 (could be done on val set, here simplified for test demonstration)
    for m in models:
        probs = m.predict(test_ds)
        preds = np.argmax(probs, axis=1)
        f1 = f1_score(y_true, preds, average="macro")
        model_f1s.append(f1)
        all_probs.append(probs)
        print(f"{m.name} F1-score: {f1:.4f}")
    
    # Weighted Average
    weights = np.array(model_f1s) / sum(model_f1s)
    ensemble_probs = np.average(all_probs, axis=0, weights=weights)
    ensemble_preds = np.argmax(ensemble_probs, axis=1)
    
    return ensemble_preds, ensemble_probs

## Execution Flow

In [ ]:
df = get_dataset()
train_df, val_df, test_df = patient_wise_split(df)

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

trained_models = []
for name in BUILDERS.keys():
    m = train_model(name, train_ds, val_ds)
    trained_models.append(m)

y_test_true = test_df.label.values
ens_preds, ens_probs = evaluate_ensemble(trained_models, test_ds, y_test_true)

print("\nENsemble Results:")
print(classification_report(y_test_true, ens_preds, target_names=Config.CLASS_NAMES.values()))
plot_confusion_matrix(y_test_true, ens_preds, "Ensemble")
plot_roc_auc(y_test_true, ens_probs, "Ensemble")

## Visualizing Model Focus (Grad-CAM)

In [ ]:
# Example for DenseNet121
sample_idx = random.randint(0, len(test_df)-1)
sample_path = test_df.iloc[sample_idx].filepath
sample_label = Config.CLASS_NAMES[test_df.iloc[sample_idx].label]

img_input, _ = parse_fn(sample_path, 0)
img_batch = tf.expand_dims(img_input, 0)

model_to_viz = trained_models[0] # DenseNet121
heatmap = get_gradcam_heatmap(model_to_viz, img_batch, "conv5_block16_concat")

print(f"Visualizing for: {sample_label}")
display_gradcam(sample_path, heatmap)